In [1]:
import os
import numpy as np
import pandas as pd
from scipy.fft import rfft, rfftfreq
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler


In [2]:
#Plan to get from Time Domain: RMS, Kurtosis, Peak amplitude, Crest Factor(Peak/RMS), Shape Factor (RMS/Mean Abs Value)
#Plan from frequency domain: Top 50 Frequencies, Magnitudes of those frequencies...
# mean freq mag, std freq mag, spectral centroid(mean freq/sum of magnitudes)

In [3]:
def process_folder(folder_path, set_id, bearing_col=0, threshold = .9):
    # Only grab files (skipping system files like desktop.ini)
    files = sorted([f for f in os.listdir(folder_path) if f not in ['desktop.ini', '.DS_Store']])
    data_list = []
    
    print(f"Processing {len(files)} files from Set {set_id}...")
    total_files = len(files)
    # We define the last 15% as the 'fault' period
    fault_threshold = int(total_files * threshold) #will be changed later
    
    for i, filename in enumerate(files):
        try:
            file_path = os.path.join(folder_path, filename)
            # NASA files use tabs; header=None because there are no column titles
            raw_data = pd.read_csv(file_path, sep='\t', header=None)
            signal = raw_data[bearing_col].values
            
            # --- 1. TIME DOMAIN ---
            rms = np.sqrt(np.mean(signal**2))
            kurtosis = pd.Series(signal).kurt()
            peak = np.max(np.abs(signal))
            crest_factor = peak / rms if rms != 0 else 0
            shape_factor = rms / np.mean(np.abs(signal)) if np.mean(np.abs(signal)) != 0 else 0
            label = "healthy" if i < fault_threshold else "fault"
            # --- 2. FREQUENCY DOMAIN ---
            fs = 20000
            yf = np.abs(rfft(signal))
            xf = rfftfreq(len(signal), 1/fs)
            
            # Global Spectral Stats
            spec_mean = np.mean(yf)
            spec_std = np.std(yf)
            spec_centroid = np.sum(xf * yf) / np.sum(yf) if np.sum(yf) != 0 else 0
            
            # Peak Finding (Top 50)
            peaks, _ = find_peaks(yf, distance=100)
            top_indices = peaks[np.argsort(yf[peaks])[-50:]][::-1]
            
            freqs = np.zeros(50)
            mags = np.zeros(50)
            freqs[:len(top_indices)] = xf[top_indices]
            mags[:len(top_indices)] = yf[top_indices]
            
            # --- 3. COMBINE ---
            # Order: ID info + 5 Time Stats + 50 Freqs + 50 Mags + 3 Global Stats
            row = [filename, set_id, label, rms, kurtosis, peak, crest_factor, shape_factor] + \
                  list(freqs) + list(mags) + \
                  [spec_mean, spec_std, spec_centroid]
            
            data_list.append(row)
            
            if i % 100 == 0:
                print(f"Processed {i}/{len(files)} files...")
                
        except Exception as e:
            print(f"Error in {filename}: {e}")
            
    return data_list

In [4]:
# 1. Set the path to your data
# For NASA Set 2, there is usually only one '2nd_test' folder containing the files
target_folder = os.path.join('raw_data', '2nd_test')

# 2. Define all columns (must match the order in process_folder exactly)
columns = ['timestamp', 'set_id', 'label', 'rms', 'kurtosis', 'peak', 'crest_factor', 'shape_factor'] + \
          [f'f_{i}' for i in range(50)] + \
          [f'm_{i}' for i in range(50)] + \
          ['spec_mean', 'spec_std', 'spec_centroid']

# 3. Run the process
# We pass bearing_col=0 to look at the first bearing in the set
processed_data = process_folder(target_folder, set_id=2, bearing_col=0)

# 4. Convert to DataFrame and Save
if processed_data:
    final_df = pd.DataFrame(processed_data, columns=columns)
    
    # --- ADD THIS FIX HERE ---
    # Convert '2004.02.12.10.32.39' to a proper datetime and then to ISO format
    final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], format='%Y.%m.%d.%H.%M.%S')
    # ISO 8601 format: YYYY-MM-DDTHH:MM:SS
    final_df['timestamp'] = final_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
    # --------------------------

    # Save to CSV
    final_df.to_csv('processed_features_set2.csv', index=False)

Processing 984 files from Set 2...
Processed 0/984 files...
Processed 100/984 files...
Processed 200/984 files...
Processed 300/984 files...
Processed 400/984 files...
Processed 500/984 files...
Processed 600/984 files...
Processed 700/984 files...
Processed 800/984 files...
Processed 900/984 files...


In [ ]:
target_folder = os.path.join('raw_data', '3rd_test')

# Define all columns for the final DataFrame
columns = ['timestamp', 'set_id', 'label', 'rms', 'kurtosis', 'peak', 'crest_factor', 'shape_factor'] + \
          [f'f_{i}' for i in range(50)] + \
          [f'm_{i}' for i in range(50)] + \
          ['spec_mean', 'spec_std', 'spec_centroid']

# Run the process
processed_data = process_folder(target_folder, set_id=3)

# Convert to DataFrame and Save
if processed_data:
    final_df = pd.DataFrame(processed_data, columns=columns)
    
    # --- ADD THIS FIX HERE ---
    # Convert '2004.02.12.10.32.39' to a proper datetime and then to ISO format
    final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], format='%Y.%m.%d.%H.%M.%S')
    # ISO 8601 format: YYYY-MM-DDTHH:MM:SS
    final_df['timestamp'] = final_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
    # --------------------------

    # Save to CSV
    final_df.to_csv('processed_features_set3.csv', index=False)

Processing 6324 files from Set 3...
Processed 0/6324 files...
Processed 100/6324 files...
Processed 200/6324 files...
Processed 300/6324 files...
Processed 400/6324 files...
Processed 500/6324 files...
Processed 600/6324 files...
Processed 700/6324 files...
Processed 800/6324 files...
Processed 900/6324 files...
Processed 1000/6324 files...
Processed 1100/6324 files...
Processed 1200/6324 files...
Processed 1300/6324 files...
Processed 1400/6324 files...
Processed 1500/6324 files...
Processed 1600/6324 files...
Processed 1700/6324 files...
Processed 1800/6324 files...
Processed 1900/6324 files...
Processed 2000/6324 files...
Processed 2100/6324 files...
Processed 2200/6324 files...
Processed 2300/6324 files...
Processed 2400/6324 files...
Processed 2500/6324 files...
Processed 2600/6324 files...
Processed 2700/6324 files...
Processed 2800/6324 files...
Processed 2900/6324 files...
Processed 3000/6324 files...
Processed 3100/6324 files...
Processed 3200/6324 files...
Processed 3300/6324

In [ ]:
target_folder = os.path.join('raw_data', '1st_test')

# Define all columns for the final DataFrame
columns = ['timestamp', 'set_id', 'label', 'rms', 'kurtosis', 'peak', 'crest_factor', 'shape_factor'] + \
          [f'f_{i}' for i in range(50)] + \
          [f'm_{i}' for i in range(50)] + \
          ['spec_mean', 'spec_std', 'spec_centroid']

# Run the process
processed_data = process_folder(target_folder, set_id=1)

# Convert to DataFrame and Save
if processed_data:
    final_df = pd.DataFrame(processed_data, columns=columns)
    
    # --- ADD THIS FIX HERE ---
    # Convert '2004.02.12.10.32.39' to a proper datetime and then to ISO format
    final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], format='%Y.%m.%d.%H.%M.%S')
    # ISO 8601 format: YYYY-MM-DDTHH:MM:SS
    final_df['timestamp'] = final_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S')
    # --------------------------

    # Save to CSV
    final_df.to_csv('processed_features_set1.csv', index=False)

Processing 2156 files from Set 1...
Processed 0/2156 files...
Processed 100/2156 files...
Processed 200/2156 files...
Processed 300/2156 files...
Processed 400/2156 files...
Processed 500/2156 files...
Processed 600/2156 files...
Processed 700/2156 files...
Processed 800/2156 files...
Processed 900/2156 files...
Processed 1000/2156 files...
Processed 1100/2156 files...
Processed 1200/2156 files...
Processed 1300/2156 files...
Processed 1400/2156 files...
Processed 1500/2156 files...
Processed 1600/2156 files...
Processed 1700/2156 files...
Processed 1800/2156 files...
Processed 1900/2156 files...
Processed 2000/2156 files...
Processed 2100/2156 files...

Done! CSV saved with ISO timestamps for Edge Impulse.


In [ ]:

# 1. Define your specific healthy-end indices, found in EE460_project_optuna.ipynb
final_indices = {
    "processed_features_set1.csv": 2095, 
    "processed_features_set2.csv": 690, 
    "processed_features_set3.csv": 6200
}

# 2. Load and combine the files
all_data_frames = []
for filename in final_indices.keys():
    # Load each set
    temp_df = pd.read_csv(filename)
    # Ensure we know which set this came from for later
    temp_df['source_file'] = filename 
    all_data_frames.append(temp_df)

# This is your 'master' dataframe
df_combined = pd.concat(all_data_frames, ignore_index=True)

# 3. Identify and extract ONLY the healthy rows to "fit" the scaler
healthy_slices = []
for filename, end_idx in final_indices.items():
    # Grab the healthy rows from the combined df for this specific file
    subset = df_combined[df_combined['source_file'] == filename].iloc[:end_idx]
    healthy_slices.append(subset)

df_healthy_only = pd.concat(healthy_slices)

# 4. Prepare for Scaling
# We need to drop non-numeric columns like 'filename', 'label', 'source_file', etc.
# Identify numeric columns (adjust names based on your CSV headers)
cols_to_drop = ['filename', 'timestamp', 'set_id', 'source_file']
numeric_healthy = df_healthy_only.drop(columns=[c for c in cols_to_drop if c in df_healthy_only.columns])
numeric_all = df_combined.drop(columns=[c for c in cols_to_drop if c in df_combined.columns])

# 5. Execute Z-Score Normalization
scaler = StandardScaler()
scaler.fit(numeric_healthy) # Learn Mean and Std from healthy data ONLY

# Transform the entire combined dataset
scaled_array = scaler.transform(numeric_all)

# 6. Create the Final Normalized DataFrame
df_normalized = pd.DataFrame(scaled_array, columns=numeric_all.columns)

# Add back the categorical info so you know which row is which
df_normalized['set_id'] = df_combined['set_id'].values
df_normalized['source_file'] = df_combined['source_file'].values

# Loop through the unique source files to split the normalized data back out
for original_name in final_indices.keys():
    # Filter the normalized dataframe for rows belonging to this specific set
    set_df = df_normalized[df_normalized['source_file'] == original_name].copy()
    
    # Drop the 'source_file' helper column so the CSV is clean for your MLP/Autoencoder
    set_df = set_df.drop(columns=['source_file'])
    
    # Define a new name (e.g., changing 'processed' to 'normalized')
    new_filename = original_name.replace("processed", "normalized")
    
    # Save the file
    set_df.to_csv(new_filename, index=False)
    print(f"Successfully saved: {new_filename}")

Successfully saved: normalized_features_set1.csv
Successfully saved: normalized_features_set2.csv
Successfully saved: normalized_features_set3.csv
